# Multiperiod Heat Integration

**Learning outcome:** Apply multiperiod heat integration through the public `PinchProblem` or `PinchWorkspace` workflow.

**Level:** Intermediate  
**Execution profile:** `base`  
**Expected runtime:** under 2 minutes  
**Optional extras:** none

The lifecycle is explicit: prepare the study, run the named method, then inspect cached results. Observation cells do not launch analysis.

## Study question and data

**Study question:** How do heat-integration targets change by operating period, and what is the weighted annual view?

The sample data is packaged with OpenPinch, so the notebook runs without path setup. Read the named inputs and assumptions before substituting plant data.

## Step 1: Run aligned period analyses

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [ ]:
from OpenPinch import PinchProblem

base_problem = PinchProblem(
    "Four-stream-Yee-and-Grossmann-1990-1.json"
)
multiperiod_input = base_problem.to_problem_json()
multiperiod_input["options"]["PROBLEM_PERIOD_IDS"] = [
    "base", "turndown"
]
multiperiod_input["options"]["PROBLEM_PERIOD_WEIGHTS"] = [
    0.7, 0.3
]
for stream in multiperiod_input["streams"]:
    duty = stream["heat_flow"]
    stream["heat_flow"] = {
        "values": [duty["value"], 0.8 * duty["value"]],
        "unit": duty["unit"],
    }
    stream["heat_capacity_flowrate"] = None
problem = PinchProblem(
    multiperiod_input, project_name="Four Stream Periods"
)
period_ids = list(problem.period_ids)
period_outputs = problem.target.all_periods.all_heat_integration()
direct_periods = problem.target.all_periods.direct_heat_integration()
indirect_periods = problem.target.all_periods.indirect_heat_integration()
site_periods = problem.target.all_periods.total_site_heat_integration()
all_periods = problem.summary_frame(include_periods=True)
assert list(period_outputs) == period_ids
assert len(period_ids) >= 2
all_periods

## Step 2: Compare period and weighted summaries

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [ ]:
weighted = problem.summary_frame(include_weighted_average=True)
combined = problem.summary_frame(
    include_periods=True, include_weighted_average=True
)
list(problem.period_results)

## Review the result

Review period results beside the weighted aggregate; the largest single-period target does not necessarily dominate the weighted study.

In [ ]:
from IPython.display import display

display(all_periods)
display(weighted)
display(combined)

## Interpret the result

Review every period before the weighted average; an annual average can conceal a limiting or infeasible operating state.

## Adapt this template

Replace the sample with period-tagged plant data and verify period order and weights before comparing annual totals.

Keep the workflow explicit: prepare input, call one named engineering method, inspect cached results, then export.